# Producer B — `camera_event_B.csv` → Kafka topic `camera-events-B`

**Author:** Haruto Iriyama (pair assignment, FIT3182 Monash University)

This notebook is **identical to `producer_a.ipynb` except for the topic, CSV file and producer id**. The full design rationale (Kafka key choice, delivery reliability via `acks='all'`/`retries=5`, batch ordering, MAX_BATCHES default) lives in producer A — read that first if you have not already.

In [1]:
# --- Imports & logging -------------------------------------------------------
# The kafka-python package may be the legacy `kafka` PyPI release whose
# simple.py uses `self.async` (a Python 3.7+ syntax error). We try the
# standard import first, then fall back to `kafka3` (the maintained fork).
import csv
import json
import logging
import os
import pathlib
import signal
import sys
import time
from collections import OrderedDict
from datetime import datetime, timezone

try:
    from kafka import KafkaProducer
    from kafka.errors import KafkaError, NoBrokersAvailable
except (ImportError, SyntaxError):
    from kafka3 import KafkaProducer
    from kafka3.errors import KafkaError, NoBrokersAvailable

logger = logging.getLogger('awas.producer.B')
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-7s | %(name)s | %(message)s'))
    logger.addHandler(h)
logger.setLevel(logging.INFO)

In [2]:
# --- Parameters --------------------------------------------------------------
HOST_IP            = '192.168.100.81'

# --- ROBUST PATH RESOLUTION ---
current_dir = pathlib.Path.cwd()
if current_dir.name == 'src':
    DATA_DIR = current_dir.parent / 'data'
else:
    DATA_DIR = current_dir / 'data'

DATA_FILE          = DATA_DIR / 'camera_event_B.csv'

assert DATA_FILE.exists(), (
    f'{DATA_FILE.resolve()} not found. Ensure the data folder is placed correctly.'
)

TOPIC              = 'camera-events-B'
PRODUCER_ID        = 'B'
BOOTSTRAP_SERVERS  = f'{HOST_IP}:9092'
BATCH_INTERVAL_SEC = 1.0  # 1 batch per second
MAX_BATCHES        = 300  # 5-min demo run; set to None to replay the entire file
# Cap emissions by EVENT-TIME so producers A/B/C cover the same camera-time
# slice regardless of how many batch_ids that contains per CSV. This keeps the
# stream-stream join in data_design_streaming.ipynb well-matched.
# Set to None to disable the event-time cap.
DEMO_EVENT_MINUTES = None

print(f'AWAS_HOST          = {HOST_IP!r}')
print(f'DATA_FILE          = {DATA_FILE.resolve()}')
print(f'TOPIC              = {TOPIC}')
print(f'BOOTSTRAP          = {BOOTSTRAP_SERVERS}')
print(f'MAX_BATCHES        = {MAX_BATCHES}')
print(f'INTERVAL_SEC       = {BATCH_INTERVAL_SEC}')
print(f'DEMO_EVENT_MINUTES = {DEMO_EVENT_MINUTES}')

AWAS_HOST          = '192.168.100.81'
DATA_FILE          = /home/student/data/camera_event_B.csv
TOPIC              = camera-events-B
BOOTSTRAP          = 192.168.100.81:9092
MAX_BATCHES        = 300
INTERVAL_SEC       = 1.0
DEMO_EVENT_MINUTES = None


In [3]:
# --- Helpers -----------------------------------------------------------------
def coerce_event_row(row, producer_id):
    """Cast raw CSV strings to a typed event payload.

    Returns None for malformed rows so the producer can log-and-skip rather
    than crash on a single bad line.

    Parameters
    ----------
    row : dict
        A row produced by csv.DictReader.
    producer_id : str
        Identifier of the producer that owns the row (e.g. 'A').

    Returns
    -------
    dict or None
        The typed payload, or None if the row was malformed.
    """
    try:
        return {
            'event_id':          row['event_id'],
            'batch_id':          int(row['batch_id']),
            'car_plate':         row['car_plate'],
            'camera_id':         int(row['camera_id']),
            'timestamp':         row['timestamp'],   # event-time at the camera
            'speed_reading':     float(row['speed_reading']),
            'producer_id':       producer_id,        # source identification
            'publish_timestamp': datetime.now(timezone.utc).isoformat(),  # wall-clock at publish
        }
    except (KeyError, ValueError) as exc:
        logger.warning('Skipping malformed row %s: %s', row, exc)
        return None


def load_batches(csv_path, producer_id):
    """Read the CSV once and group rows by batch_id, preserving file order.

    Parameters
    ----------
    csv_path : pathlib.Path
        Path to the camera event CSV.
    producer_id : str
        Identifier of the producer (used to label every event).

    Returns
    -------
    collections.OrderedDict
        Mapping batch_id -> list[event_dict], in original file order.
    """
    batches = OrderedDict()
    with csv_path.open('r', encoding='utf-8', newline='') as fh:
        for raw_row in csv.DictReader(fh):
            ev = coerce_event_row(raw_row, producer_id)
            if ev is None:
                continue
            batches.setdefault(ev['batch_id'], []).append(ev)
    logger.info('Loaded %d batches (%d events) from %s',
                len(batches),
                sum(len(b) for b in batches.values()),
                csv_path.name)
    return batches


def build_producer(bootstrap_servers, client_id):
    """Build a hardened KafkaProducer: acks=all, reliable delivery, JSON serialiser.

    Parameters
    ----------
    bootstrap_servers : str
        Kafka bootstrap address, e.g. 'localhost:9092'.
    client_id : str
        Distinct client id for this producer (helps Kafka diagnostics).

    Returns
    -------
    KafkaProducer
        Configured producer ready to send.
    """
    return KafkaProducer(
        bootstrap_servers=bootstrap_servers,
        client_id=client_id,
        acks='all',                  # wait for all in-sync replicas before acking
        retries=5,                   # retry transient errors
        linger_ms=20,                # micro-batch on the network side
        # enable_idempotence is not supported by kafka3; reliability is
        # achieved via acks='all' and retries=5 (identical durability guarantee).
        key_serializer=lambda k: k.encode('utf-8') if k else None,
        value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    )


def on_send_error(excp, batch_id, producer_id):
    """Log every Kafka send failure so the operator notices."""
    logger.error('[%s] send failed (batch=%s): %s', producer_id, batch_id, excp)

In [4]:
# --- Load CSV ----------------------------------------------------------------
batches = load_batches(DATA_FILE, PRODUCER_ID)

# Optional event-time cap: keep only events whose CSV `timestamp` falls within
# the first DEMO_EVENT_MINUTES of event-time. All three producers apply the
# same cutoff so their A/B/C streams overlap on the same camera-time window
# and the downstream stream-stream join matches more pairs.
if DEMO_EVENT_MINUTES is not None:
    from datetime import timedelta
    all_ts   = [datetime.fromisoformat(ev['timestamp'])
                for evs in batches.values() for ev in evs]
    start_ts = min(all_ts)
    cutoff   = start_ts + timedelta(minutes=DEMO_EVENT_MINUTES)
    filtered = OrderedDict()
    for bid, evs in batches.items():
        kept = [ev for ev in evs
                if datetime.fromisoformat(ev['timestamp']) <= cutoff]
        if kept:
            filtered[bid] = kept
    n_before = sum(len(b) for b in batches.values())
    n_after  = sum(len(b) for b in filtered.values())
    logger.info('[demo mode] keeping %s min of event-time (%s -> %s); '
                'kept %d / %d events across %d / %d batches',
                DEMO_EVENT_MINUTES, start_ts, cutoff,
                n_after, n_before, len(filtered), len(batches))
    batches = filtered

first_id = next(iter(batches))
import pprint
print(f'First batch id: {first_id}; sample event:')
pprint.pprint(batches[first_id][0])

2026-05-25 15:34:58,525 | INFO    | awas.producer.B | Loaded 294206 batches (556340 events) from camera_event_B.csv
First batch id: 1; sample event:
{'batch_id': 1,
 'camera_id': 2,
 'car_plate': 'CJW 924',
 'event_id': 'b5b7dae7-3bf9-4f75-aea8-398d3bde3a41',
 'producer_id': 'B',
 'publish_timestamp': '2026-05-25T15:34:47.725837+00:00',
 'speed_reading': 156.8,
 'timestamp': '2024-01-01T08:00:26.029536'}


In [ ]:
# --- Run the producer --------------------------------------------------------
try:
    producer = build_producer(BOOTSTRAP_SERVERS, client_id=f'awas-producer-{PRODUCER_ID}')
except NoBrokersAvailable as exc:
    raise RuntimeError(
        f'Could not reach Kafka at {BOOTSTRAP_SERVERS}. '
        f'Check that the Kafka container is running and AWAS_HOST is set correctly.'
    ) from exc

def _sigint(signum, frame):
    logger.info('Producer %s received SIGINT, flushing...', PRODUCER_ID)
    producer.flush(timeout=10)
    producer.close(timeout=10)
    sys.exit(0)
signal.signal(signal.SIGINT, _sigint)

sent_batches = sent_events = 0
try:
    for batch_id, events in batches.items():
        if MAX_BATCHES is not None and sent_batches >= MAX_BATCHES:
            logger.info('MAX_BATCHES=%s reached, stopping.', MAX_BATCHES)
            break
        for ev in events:
            # key=car_plate ensures all events for the same vehicle land on
            # the same Kafka partition, preserving per-vehicle ordering.
            fut = producer.send(TOPIC, key=ev['car_plate'], value=ev)
            fut.add_errback(on_send_error, batch_id=batch_id, producer_id=PRODUCER_ID)
        producer.flush(timeout=30)
        sent_batches += 1
        sent_events += len(events)
        logger.info('[%s] published batch_id=%s (%d events) | totals: %d batches, %d events',
                    PRODUCER_ID, batch_id, len(events), sent_batches, sent_events)
        time.sleep(BATCH_INTERVAL_SEC)
finally:
    producer.flush(timeout=30)
    producer.close(timeout=30)
    logger.info('[%s] Closed producer cleanly. Sent %d batches, %d events.',
                PRODUCER_ID, sent_batches, sent_events)

2026-05-25 15:34:58,691 | INFO    | awas.producer.B | [B] published batch_id=1 (5 events) | totals: 1 batches, 5 events
2026-05-25 15:34:59,704 | INFO    | awas.producer.B | [B] published batch_id=2 (1 events) | totals: 2 batches, 6 events
2026-05-25 15:35:00,719 | INFO    | awas.producer.B | [B] published batch_id=3 (1 events) | totals: 3 batches, 7 events
2026-05-25 15:35:01,728 | INFO    | awas.producer.B | [B] published batch_id=4 (4 events) | totals: 4 batches, 11 events
2026-05-25 15:35:02,824 | INFO    | awas.producer.B | [B] published batch_id=5 (4 events) | totals: 5 batches, 15 events
2026-05-25 15:35:03,837 | INFO    | awas.producer.B | [B] published batch_id=6 (2 events) | totals: 6 batches, 17 events
2026-05-25 15:35:04,845 | INFO    | awas.producer.B | [B] published batch_id=7 (3 events) | totals: 7 batches, 20 events
2026-05-25 15:35:05,853 | INFO    | awas.producer.B | [B] published batch_id=8 (1 events) | totals: 8 batches, 21 events
2026-05-25 15:35:06,872 | INFO    |